In [2]:
#여러 라이브러리를 플러그인으로 추가할 수 있음
!pip install llama-index==0.11.11

In [3]:
#크로마를 라마인덱스의 벡터 저장소로 사용할 수 있도록 해주는 플러그인
!pip install llama-index-vector-stores-chroma==0.3.0

In [4]:
#허깅페이스의 임베딩 모델을 라마인덱스에서 쉽게 활용할 수 있도록 해줌
!pip install llama-index-embeddings-huggingface==0.3.0

In [5]:
import chromadb
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os
api_key = os.environ.get("OPENAI_API_KEY") 

In [6]:
#chromaDB 클라이언트 생성 및 컬렉션 준비, 데이터를 저장할 로컬 경로 지정
client = chromadb.PersistentClient(path="./chroma_db")
#컬렉션 생성 또는 불러오기
collection = client.get_or_create_collection("example-collection")

#huggingFace 임베딩 모델 설정
embed_model = HuggingFaceEmbedding(model_name="all-MiniLm-L6-v2")

#문서 데이터 준비
documents = [
    "고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.",
    "강아지는 충성심이 강하고 친절한 동물로, 흔히 인간의 최고의 친구로 불립니다. 주로 애완동물로 기르고, 동반자로 유명합니다.",
    "고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다."
]
ids = ["doc1", "doc2", "doc3"]

#문서를 llamaIndex의 Document 형식으로 변환
nodes = [Document(text=doc, id_=doc_id) for doc, doc_id in zip(documents, ids)]

In [7]:
#문서를 크로마 벡터 스토어에 적재
import os
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.llms.openai import OpenAI

llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

#Chroma 벡터 스토어 생성
vector_store = ChromaVectorStore(chroma_collection=collection)

#llamaIndex의 VectorStoreIndex 생성
#문서 데이터를 벡터화하고, 크로마로 인덱싱함
index = VectorStoreIndex.from_documents(nodes, vector_store=vector_store, embed_model= embed_model, llm=llm)


In [8]:
#쿼리 엔진 생성
query_engine = index.as_query_engine()

#질의 수행
query_text = "고양이에 대해 알려줘"
response = query_engine.query(query_text)

#결과 출력
print("[질의 결과]")
print(response)

[질의 결과]
고양이는 강아지와 마찬가지로 전 세계적으로 인기 있는 애완동물 중 하나입니다. 고양이는 독립적이고 까다로운 성격을 가지고 있으며, 주로 깨끗한 동물로 알려져 있습니다. 또한 사냥본능이 강하고 조용한 성격을 가지고 있습니다.


In [1]:
from llama_index.core import Settings

In [13]:
#llamaindex = VectorStoreIndex 생성
#embed_model 지정안하면 기본으로 openAI text-embedding-ada-002 모델 사용
index = VectorStoreIndex.from_documents(nodes, vector_store=vector_store)

#쿼리 엔진 생성(기본적인 검색 + 답변 생성 기능 활성화
query_enging = index.as_query_engine()

query_text = "고양이에 대해 알려줘"
response = query_enging.query(query_text)

#최종 응답 출력
print("[질의 결과]")
print(response)

#응답 생성에 사용된 문서 확인
print("\n[검색 문서]")
for i, node in enumerate(response.source_nodes, 1): 
    print(f"{i}. {node.text}\n")

[질의 결과]
고양이는 작은 육식동물로, 주로 애완동물로 기르며 민첩하고 장난기 있는 행동으로 유명합니다.

[검색 문서]
1. 고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.

2. 고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다.

